# GRU4Rec -- Untrained Embedding Generation (GPU)

Untrained embedding generation -- GRU4Rec (GPU)

- No training: the model is built with random starting weights and run
  through the data exactly ONCE (`torch.no_grad()`). No loss, no
  optimizer, no epochs.
- Input: `data/bucketized_30day_all_events.parquet` (already generated --
  this notebook only reads it, never recreates it). This is a self-
  contained copy inside this folder -- nothing outside `standalone_embeddings/`
  is required.
- Output: `gru4rec_embeddings.parquet` in this same folder -- one row per
  (user, event) pair, 256-dimensional embedding vector each.
- Device: forced to **GPU** in this version.


In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn


In [ ]:
LOW_CARD_COLUMNS = [
    "channel",
    "device_type",
    "edition",
    "georegion",
    "page_hit_ref_type",
    "paywall",
    "content_source_hit",
    "content_type_hit",
    "section_hit",
    "subscriber_status",
]
HIGH_CARD_COLUMNS = ["page_id"]
BEHAVIOR_COLUMNS = LOW_CARD_COLUMNS + HIGH_CARD_COLUMNS

LOW_CARD_DIM = 192
HIGH_CARD_DIM = 64
DIM = LOW_CARD_DIM + HIGH_CARD_DIM  # 256
MIN_COL_DIM = 8
NUM_LAYERS = 2
NUM_HEADS = 4
PAD_IDX = 0
UNK_IDX = 1
RESERVED = 2
BATCH_SIZE = 128

INPUT_PATH = "../data/bucketized_30day_all_events.parquet"
OUTPUT_PATH = "gru4rec_embeddings.parquet"


In [ ]:
assert torch.cuda.is_available(), "CUDA GPU not available in this environment -- use the CPU notebook instead"
device = torch.device("cuda")
print(f"device: {device} ({torch.cuda.get_device_name(0)})")


In [ ]:
class TokenVocab:
    """Same PAD=0 / UNK=1 convention used throughout this project."""

    def __init__(self):
        self.token_to_idx = {}

    def fit(self, values):
        for v in values:
            v = str(v)
            if v not in self.token_to_idx:
                self.token_to_idx[v] = len(self.token_to_idx) + RESERVED
        return self

    def transform(self, values):
        return [self.token_to_idx.get(str(v), UNK_IDX) for v in values]

    def __len__(self):
        return len(self.token_to_idx) + RESERVED


def assign_column_dims(vocab_sizes: dict, total_dim: int, min_dim: int) -> dict:
    """Cardinality-proportional allocation with a floor -- a column with
    more distinct values gets more room, subject to a MIN_COL_DIM floor so
    no column gets squeezed into too little space."""
    cols = sorted(vocab_sizes, key=lambda c: vocab_sizes[c])
    n = len(cols)
    remaining = total_dim - min_dim * n
    if remaining < 0:
        raise ValueError(f"total_dim={total_dim} too small for {n} columns at min_dim={min_dim}")
    total_cardinality = sum(vocab_sizes.values())
    dims, running_extra = {}, 0
    for i, col in enumerate(cols):
        if i < n - 1:
            extra = round(remaining * vocab_sizes[col] / total_cardinality)
            running_extra += extra
        else:
            extra = remaining - running_extra
        dims[col] = min_dim + extra
    return dims


In [ ]:
def encode_batch(chunk: pd.DataFrame, vocabs: dict, device: torch.device):
    """Turns one chunk of rows into padded tensors, sized to this
    chunk's own local max sequence length (not the whole dataset's global
    max) -- no length cap, full real window used."""
    B = len(chunk)
    L = max(int(chunk["n_hits_in_window"].max()), 1)

    item_idx = {col: np.zeros((B, L), dtype=np.int64) for col in BEHAVIOR_COLUMNS}
    timestamps = np.zeros((B, L), dtype=np.int64)
    padding_mask = np.zeros((B, L), dtype=bool)

    for b, row in enumerate(chunk.itertuples()):
        ts_list = getattr(row, "date_time_et_30_days")
        n = min(len(ts_list), L)
        timestamps[b, :n] = ts_list[:n]
        padding_mask[b, :n] = True
        for col in BEHAVIOR_COLUMNS:
            vals = getattr(row, f"{col}_30_days")[:n]
            item_idx[col][b, :n] = vocabs[col].transform(vals)

    return (
        {col: torch.tensor(item_idx[col], dtype=torch.long, device=device) for col in BEHAVIOR_COLUMNS},
        torch.tensor(timestamps, dtype=torch.float32, device=device),
        torch.tensor(padding_mask, dtype=torch.bool, device=device),
    )


In [ ]:
df = pd.read_parquet(INPUT_PATH)
n_events = len(df)
global_max_len = max(int(df["n_hits_in_window"].max()), 1)
print(f"(user, event) rows: {n_events}, unique users: {df['post_evar3'].nunique()}, "
      f"max hits in any window: {global_max_len}, batch size: {BATCH_SIZE}")

vocabs = {}
for col in BEHAVIOR_COLUMNS:
    all_vals = [v for lst in df[f"{col}_30_days"] for v in lst]
    vocabs[col] = TokenVocab().fit(all_vals)

vocab_sizes = {col: len(vocabs[col]) for col in BEHAVIOR_COLUMNS}
low_card_vocab_sizes = {col: vocab_sizes[col] for col in LOW_CARD_COLUMNS}
col_dims = assign_column_dims(low_card_vocab_sizes, LOW_CARD_DIM, MIN_COL_DIM)
col_dims["page_id"] = HIGH_CARD_DIM
assert sum(col_dims.values()) == DIM

print("column sizing (vocab size -> assigned embedding dim):")
for col in BEHAVIOR_COLUMNS:
    print(f"  {col}: {vocab_sizes[col]} distinct values (incl. PAD/UNK) -> {col_dims[col]}-d")


In [ ]:
class GRU4RecEncoderModel(nn.Module):
    """No attention at all -- a GRU reads the sequence step by step;
    its final hidden state IS the summary vector."""

    def __init__(self, vocab_sizes: dict, col_dims: dict, max_len: int):
        super().__init__()
        self.embeddings = nn.ModuleDict(
            {col: nn.Embedding(vocab_sizes[col], col_dims[col], padding_idx=PAD_IDX) for col in BEHAVIOR_COLUMNS}
        )
        self.combine_proj = nn.Linear(DIM, DIM)
        self.gru = nn.GRU(DIM, DIM, num_layers=NUM_LAYERS, batch_first=True)
        self.final_norm = nn.LayerNorm(DIM)

    def forward(self, item_idx_t, timestamps_t, padding_mask_t):
        parts = [self.embeddings[col](item_idx_t[col]) for col in BEHAVIOR_COLUMNS]
        x = self.combine_proj(torch.cat(parts, dim=-1))

        lengths = padding_mask_t.sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        summary = self.final_norm(hidden[-1])  # last layer's final state
        return summary  # (B, DIM)


torch.manual_seed(42)  # fixed seed -- makes the untrained random weights REPRODUCIBLE across runs
model = GRU4RecEncoderModel(vocab_sizes, col_dims, global_max_len).to(device)
model.eval()


In [ ]:
all_vectors = []
n_batches = (n_events + BATCH_SIZE - 1) // BATCH_SIZE
with torch.no_grad():
    for i, start in enumerate(range(0, n_events, BATCH_SIZE)):
        chunk = df.iloc[start:start + BATCH_SIZE]
        item_idx_t, timestamps_t, padding_mask_t = encode_batch(chunk, vocabs, device)
        summary = model(item_idx_t, timestamps_t, padding_mask_t)
        all_vectors.append(summary.cpu().numpy())
        print(f"  batch {i + 1}/{n_batches}: {len(chunk)} rows, local max len {timestamps_t.shape[1]}")

event_vectors = np.concatenate(all_vectors, axis=0)  # (n_events, DIM)
print(f"generated {event_vectors.shape[0]} embeddings of dim {event_vectors.shape[1]}")


In [ ]:
out_rows = []
for b in range(n_events):
    row = {"post_evar3": df["post_evar3"].iloc[b], "event_date": df["event_date"].iloc[b]}
    for d in range(DIM):
        row[f"emb_{d}"] = event_vectors[b, d]
    out_rows.append(row)

out_df = pd.DataFrame(out_rows)
out_df.to_parquet(OUTPUT_PATH, index=False)
print(f"wrote {len(out_df)} embeddings ({DIM}-d) -> {OUTPUT_PATH}")


In [ ]:
# quick sanity check
emb_cols = [c for c in out_df.columns if c.startswith("emb_")]
print("shape:", out_df.shape)
print("NaNs in embedding columns:", out_df[emb_cols].isna().sum().sum())
print("duplicate (post_evar3, event_date) pairs:", out_df.duplicated(subset=["post_evar3", "event_date"]).sum())
out_df.head(3)
